# CASA extraction UI (Colab)

Runs the DOC-01 extractor with a web page to upload a form. Use a T4 GPU: Runtime > Change runtime type. Restart the session first, then run every cell in order.

The last cell prints a public link. Open it, upload a form, and check the fields marked REVIEW.

In [ ]:
!rm -rf /content/repo
!git clone -q -b feature/casa-doc01-rules https://github.com/Devadn11206/DOC_Extractor_QUAPT.git /content/repo
!cd /content/repo && git log --oneline -1

In [ ]:
import subprocess
print('Code version:', subprocess.run(['git','-C','/content/repo','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())
ui = open('/content/repo/src/doc_extractor/casa_ui.py', encoding='utf-8').read()
ins = open('/content/repo/src/doc_extractor/insurance_reader.py', encoding='utf-8').read()
assert 'def make_app' in ui and 'nominee_rows' in ui, 'OLD UI CODE: restart the session and re-run the clone cell.'
assert 'insurance_fields_from_text' in ins, 'OLD INSURANCE CODE: restart the session and re-run the clone cell.'
print('OK: latest UI and insurance code present')

In [ ]:
!pip install -q -r /content/repo/requirements.txt rapidocr_onnxruntime accelerate bitsandbytes gradio
import transformers
print('transformers', transformers.__version__)

In [ ]:
import sys, types, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s', force=True)
pkg = types.ModuleType('doc_extractor')
pkg.__path__ = ['/content/repo/src/doc_extractor']
sys.modules['doc_extractor'] = pkg
from doc_extractor.casa_reader import QwenVLCasaReader
from doc_extractor.casa_ui import make_app
reader = QwenVLCasaReader(model_id='Qwen/Qwen2.5-VL-7B-Instruct', load_in_4bit=True)
demo = make_app(reader)
print('UI ready')

In [ ]:
demo.launch(share=True, debug=True)  # debug=True shows errors and progress in this cell
